In [ ]:
import tensorflow as tf
from tensorflow import keras
#import matplotlib.pyplot as plt
import numpy as np

In [ ]:
dataset = tf.keras.utils.image_dataset_from_directory(
    "E:/Paddy_detect/archive",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

In [ ]:
class_names=dataset.class_names
print(class_names)

In [ ]:
data_dir = "E:/Paddy_detect/archive"

In [ ]:
normalization_layer=keras.layers.Rescaling(2./255)

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir ,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

In [ ]:
val_ds

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
model = keras.Sequential([
    normalization_layer,

    keras.layers.Conv2D(32, 3, activation='relu'),
    keras.layers.MaxPooling2D(),

    keras.layers.Conv2D(64, 3, activation='relu'),
    keras.layers.MaxPooling2D(),

    keras.layers.Conv2D(128, 3, activation='relu'),
    keras.layers.MaxPooling2D(),

    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(len(class_names), activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
img = tf.keras.utils.load_img(
    "archive/tungro/100461.jpg",
    target_size=(224, 224)
)

In [ ]:
img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)

In [ ]:
prediction=model.predict(img_array)

In [ ]:
score = tf.nn.softmax(prediction[0])

print("Disease:",
      class_names[np.argmax(score)])

print("Confidence:",
      100 * np.max(score), "%")

In [ ]:
model.save("paddy_disease_model.h5")